In [35]:
import sys
import torch
import functools
import matplotlib.pyplot as plt
import argparse, yaml, os
import torch.optim.lr_scheduler as lr_scheduler
import torchvision.transforms as transforms
import seaborn as sns
import pandas as pd
import tqdm
import glob

from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from types import SimpleNamespace

from IPython.display import clear_output

sys.path.append('/om2/user/jmhicks/projects/TextureStreaming/code/')

from chexture_choolbox.auditorytexture.statistics_sets import (
    STAT_SET_FULL_MCDERMOTTSIMONCELLI as statistics_dict
)
from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params as model_params_tm
from texture_prior.params import statistics_set, texture_dataset
from texture_prior.utils import path

sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/om2/user/bjmedina/auditory-memory/memory/")
import DistanceMemoryModel
import encoders

from utls.loading import load_results, load_results_with_isi0_exclusion, load_results_with_isi0_dprime_exclusion, move_sequences_to_used, load_results_with_exclusion
from utls.dprime import recompute_dprime_by_isi_per_subject
from utls.reliability import compute_itemwise_split_half_reliability
from utls.plotting import plot_dprime_by_isi, plot_itemwise_split_half_scatter_df, ensure_dir


sounds_list = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")
device = 'cuda'
# get soem textures
pc_dims = 256

pc_texture_model = encoders.AudioTextureEncoderPCA(
    statistics_dict=statistics_set.statistics,
    pc_dims=pc_dims,
    model_params=model_params_tm,
    sr=20000,
    rms_level=0.05,
    duration=2.0,
    device=device
)

def compute_likelihood(score_model, input_stats, ckpt):
    """Computing the actual "prior" value
    If the score function can be treated as the vector field govering the temporal evolution
    of $x_t$ then we can integrate the ODE and apply a change of basis to evaluate the prior
    """
    score_model.load_state_dict(torch.load(ckpt))
    input_stats = input_stats.to(device)

    _, bpd = ode_likelihood(input_stats, score_model, marginal_prob_std_fn,
                            diffusion_coeff_fn,
                            input_stats.shape[0], device=device, eps=1e-5)
    
    return bpd

def parse(d):
  x = SimpleNamespace()
  _ = [setattr(x, k, parse(v)) if isinstance(v, dict) else setattr(x, k, v) for k, v in d.items() ]    
  return x

In [36]:
# parser.add_argument('--config', type=str, required=True, help='model + data configuration')
# parser.add_argument('--train', action='store_true', help='train the score-based model')
# parser.add_argument('--sample', action='store_true', help='sample from a trained model')
# parser.add_argument('--likelihood_eval', action='store_true', help='evaluate the data likelihood')
# parser.add_argument('--restart', action='store_true')
# parser.add_argument('--mode', type=str, default='textures', choices=['textures', 'mixtures'])

# sys.path.append('/om2/user/lakshmin/audio-prior/')
# from models import ScoreNetAudio, ScoreNetTexture1D, ScoreNetAudioV2
# from utils import synthesis, projection, audio
# from utils.sde_utils import *

import sys
import importlib.util
import os

# Add the new path
audio_prior_path = '/om2/user/lakshmin/audio-prior/'
sys.path.insert(0, audio_prior_path)  # insert at front of sys.path

from models import ScoreNetAudio, ScoreNetTexture1D, ScoreNetAudioV2
from utils import synthesis, projection, audio
from utils.sde_utils import *

config = "/om2/user/bjmedina/auditory-memory/memory/assets/bryan.yaml"
train = False
sample = False
likelihood_eval = True
restart = False
mode = 'textures'

device = 'cuda'

In [37]:
df = yaml.safe_load(open(config))
cfg = parse(df)

print(cfg.data)

namespace(data_root='/om2/user/lakshmin/audio-prior/assets/texture_statistics_4096texturePCs.pt', data_mix_root='/om2/user/lakshmin/audio-prior/assets/mixture_statistics_4096PCs.pt', n_pcs=256, var_scale=False)


In [38]:
score_model = torch.nn.DataParallel(
                    ScoreNetAudioV2(
                        marginal_prob_std=marginal_prob_std_fn, 
                        channels=cfg.model.channels, 
                        embed_dim=cfg.model.embed_dim
                        )
                    )

score_model = score_model.to(device)

ckpt_path = cfg.model.ckpt_path.format(cfg.data.n_pcs, mode)
ckpt_path = "/om2/user/lakshmin/audio-prior/" + ckpt_path
if 'SLURM_RESTART_COUNT' in os.environ.keys() or restart:
    score_model.load_state_dict(torch.load(ckpt_path))

print(ckpt_path)

/om2/user/lakshmin/audio-prior/ckpts/texture_diffusion2D_prior_256pcs_mode_textures.pth


In [40]:
# from dataloader import TextureStatsDataset
# texture_dataset = TextureStatsDataset(
#                 config=cfg,                        
#                 device=device
#             )
# x = texture_dataset.get_random_batch(1024)


sounds_to_test = 2000
effective_num_sounds = sounds_to_test

rep = []

t_something = 0.01

exp_sounds = glob.glob("/mindhive/mcdermott/www/mturk_stimuli/bjmedina/mem_exp_atexts_2025/*wav")

for sound in exp_sounds:
    vec = pc_texture_model(sound)

    try:
        if not torch.is_tensor(vec):
            vec = torch.tensor(vec)  # convert if it's a numpy array
        rep.append(vec)

        t = torch.tensor([t_something], device=vec.device).float()

        

        score = score_model(vec.view(1, 1, 1, 256), t)

        # 2. Flatten for norm computation per sample
        score_flat = score.view(score.size(0), -1)  # [B, 256]
        
        # 3. Normalize: divide by norm per sample
        norms = score_flat.norm(p=2, dim=1, keepdim=True) + 1e-8  # prevent division by zero
        unit_score_flat = score_flat / norms  # [B, 256]

        #print(unit_score_flat)
        
        # 4. Reshape back to original format
        unit_score = unit_score_flat.view_as(score)  # [B, 1, 1, 256]

        #input()
        #clear_output(wait=True)

    
    except Exception as e:
        print(e)
        effective_num_sounds -= 1


# Now stack all into one tensor
x = torch.stack(rep).float()  # shape: [N, ...]
x = x.view(len(exp_sounds), 1, 1, 256)

bpd_textures = compute_likelihood(score_model, input_stats=x, ckpt=ckpt_path)

x_numpy = x.squeeze(1).squeeze(1).detach().cpu().numpy()


In [42]:
x_orig = x_numpy
idx = np.random.random_integers(0, x_orig.shape[0]-1, cfg.sample.sample_batch_size)
x_orig = x_orig[idx]

sample_batch_size = cfg.sample.sample_batch_size
sampler = Euler_Maruyama_sampler_1d if cfg.model.use_single_dim_conv else Euler_Maruyama_sampler
init_dims = [1, cfg.data.n_pcs] if cfg.model.use_single_dim_conv else [1, 1, cfg.data.n_pcs]

# generate samples using the specified sampler.
samples, trajectory = sampler(
                        score_model,
                        marginal_prob_std_fn,
                        diffusion_coeff_fn,
                        num_steps=cfg.sample.num_steps,
                        batch_size=sample_batch_size,
                        device=device,
                        init_dims=init_dims)

samples = samples.squeeze().detach().cpu().numpy()


/tmp/ipykernel_424434/29064325.py:2: DeprecationWarning: This function is deprecated. Please call randint(0, 79 + 1) instead
  idx = np.random.random_integers(0, x_orig.shape[0]-1, cfg.sample.sample_batch_size)
100%|██████████| 2500/2500 [01:20<00:00, 30.96it/s]


In [ ]:
# ── Trajectory snapshots: noise → mid → sample (projected to PC1/PC2) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# --- 1) Project all trajectories to 2D using existing PCA basis (mu, W from above) ---
# trajectory is a list of tensors, each (num_steps, 1, 1, 256)
num_samples = len(trajectory)
num_steps = trajectory[0].shape[0]

# Stack and squeeze to (num_samples, num_steps, 256), then project
traj_np = np.stack([
    t.squeeze(1).squeeze(1).detach().cpu().numpy()
    if hasattr(t, 'detach') else t.squeeze(1).squeeze(1).numpy()
    for t in trajectory
])  # (num_samples, num_steps, 256)

# Project: (x - mu) @ W.T  →  (num_samples, num_steps, 2)
traj_2d = (traj_np - mu) @ W.T

# Training data in 2D (Z already exists from cell above, but recompute for safety)
Z_train = (X - mu) @ W.T  # (N_train, 2)

# --- 2) Pick timestep indices ---
step_indices = [0, num_steps // 2, num_steps - 1]
step_labels = [
    f"Step 0 (noise)",
    f"Step {num_steps // 2} (mid)",
    f"Step {num_steps - 1} (sample)",
]

# --- 3) Pick a few highlight trajectories to draw full paths ---
np.random.seed(42)
n_highlight = 5
highlight_idx = np.random.choice(num_samples, size=n_highlight, replace=False)
highlight_colors = plt.cm.tab10(np.linspace(0, 1, n_highlight))

# --- 4) KDE contours for training data background ---
kde_train = gaussian_kde(Z_train.T, bw_method=0.15)

# --- 5) Plot ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)

for panel, (step_i, label) in enumerate(zip(step_indices, step_labels)):
    ax = axes[panel]

    # Background: KDE contours of training data
    ax.imshow(Q, origin="lower", cmap="YlOrRd", alpha=0.35,
              extent=[z0.min(), z0.max(), z1.min(), z1.max()], aspect="equal")
    ax.contour(G0, G1, Q, levels=8, colors="k", linewidths=0.5, alpha=0.4)

    # Training data scatter
    ax.scatter(Z_train[:, 0], Z_train[:, 1], c="gray", s=8, alpha=0.3, label="Training data")

    # All sample positions at this timestep
    ax.scatter(traj_2d[:, step_i, 0], traj_2d[:, step_i, 1],
               c="steelblue", s=6, alpha=0.4, label="Samples")

    # Highlight trajectories: full path up to this timestep
    for k, hi in enumerate(highlight_idx):
        path = traj_2d[hi, :step_i + 1, :]
        ax.plot(path[:, 0], path[:, 1], color=highlight_colors[k],
                linewidth=1.0, alpha=0.7)
        # Mark current position
        ax.scatter(path[-1, 0], path[-1, 1], color=highlight_colors[k],
                   s=50, edgecolors="k", linewidths=0.5, zorder=5)

    ax.set_title(label, fontsize=16)
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.set_xlim(z0.min(), z0.max()); ax.set_ylim(z1.min(), z1.max())
    if panel == 0:
        ax.legend(loc="lower right", fontsize=8)

fig.suptitle("Diffusion trajectory snapshots (PC1 vs PC2)", fontsize=18, y=1.02)
plt.savefig("trajectory_snapshots.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Plotted {num_samples} samples across {num_steps} steps, "
      f"with {n_highlight} highlighted trajectories.")

In [ ]:
# ── Animated trajectory: noise → data (PC1/PC2) ──
import matplotlib.animation as animation
from IPython.display import HTML

# --- Settings ---
subsample_every = 16           # show every 16th step (4096 → 256 frames)
trail_length = 30              # number of past (subsampled) steps to show as trail
n_highlight_anim = 5           # number of trajectories to highlight with trails
save_path_mp4 = "trajectory_animation.mp4"

# --- Subsample the projected trajectories ---
frame_indices = np.arange(0, num_steps, subsample_every)
n_frames = len(frame_indices)
traj_sub = traj_2d[:, frame_indices, :]  # (num_samples, n_frames, 2)

# Pick highlight trajectories (reuse from snapshot cell or pick new ones)
np.random.seed(42)
hl_idx = np.random.choice(num_samples, size=n_highlight_anim, replace=False)
hl_colors = plt.cm.tab10(np.linspace(0, 1, n_highlight_anim))

# --- Set up figure ---
fig, ax = plt.subplots(figsize=(7, 6))

# Static background: KDE + training data
extent = [z0.min(), z0.max(), z1.min(), z1.max()]
ax.imshow(Q, origin="lower", cmap="YlOrRd", alpha=0.3, extent=extent, aspect="equal")
ax.contour(G0, G1, Q, levels=8, colors="k", linewidths=0.5, alpha=0.35)
ax.scatter(Z_train[:, 0], Z_train[:, 1], c="gray", s=5, alpha=0.2, zorder=1)
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
ax.set_xlabel("PC1"); ax.set_ylabel("PC2")

# Dynamic artists
sc_all = ax.scatter([], [], c="steelblue", s=4, alpha=0.3, zorder=2)
sc_hl = [ax.scatter([], [], color=hl_colors[k], s=60, edgecolors="k",
                     linewidths=0.5, zorder=5) for k in range(n_highlight_anim)]
trail_lines = []  # will hold Line2D objects for fading trails
step_text = ax.text(0.02, 0.97, "", transform=ax.transAxes, fontsize=12,
                    va="top", ha="left", bbox=dict(boxstyle="round,pad=0.3",
                    facecolor="white", alpha=0.8))

def init():
    sc_all.set_offsets(np.empty((0, 2)))
    for s in sc_hl:
        s.set_offsets(np.empty((0, 2)))
    step_text.set_text("")
    return [sc_all, step_text] + sc_hl

def update(frame):
    # Remove old trail lines
    for ln in trail_lines:
        ln.remove()
    trail_lines.clear()

    # All sample positions at this frame
    pts = traj_sub[:, frame, :]  # (num_samples, 2)
    sc_all.set_offsets(pts)

    # Highlight trajectories: fading trails + current dot
    trail_start = max(0, frame - trail_length)
    for k, hi in enumerate(hl_idx):
        # Current position
        sc_hl[k].set_offsets(traj_sub[hi, frame, :].reshape(1, 2))

        # Trail: draw line segments with fading alpha
        if frame > trail_start:
            trail_pts = traj_sub[hi, trail_start:frame + 1, :]
            n_seg = len(trail_pts) - 1
            for seg_i in range(n_seg):
                alpha = 0.1 + 0.6 * (seg_i / max(n_seg, 1))
                ln, = ax.plot(
                    trail_pts[seg_i:seg_i + 2, 0],
                    trail_pts[seg_i:seg_i + 2, 1],
                    color=hl_colors[k], linewidth=1.5, alpha=alpha, zorder=4
                )
                trail_lines.append(ln)

    orig_step = frame_indices[frame]
    step_text.set_text(f"Step {orig_step} / {num_steps - 1}")

    return [sc_all, step_text] + sc_hl + trail_lines

ani = animation.FuncAnimation(
    fig, update, init_func=init,
    frames=n_frames, interval=40, blit=True
)

# Save to mp4
ani.save(save_path_mp4, dpi=150, fps=25,
         writer=animation.FFMpegWriter(fps=25, bitrate=2000))
print(f"Animation saved to {save_path_mp4}")

# Display inline
HTML(ani.to_jshtml())